## Import Libraries

In [1]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import os
from threading import Thread
from concurrent.futures import ThreadPoolExecutor, wait, ALL_COMPLETED
import pandas as pd

## Load Crop Count Original Data

In [3]:
annotations_df = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_Annotations.csv")

In [4]:
annotations_df["wm_gm_ratio"]=annotations_df["WM_count"]/annotations_df["GM_count"]

In [5]:
annotations_df[annotations_df["wm_gm_ratio"]>1.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,2.446171
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912
13,13,14_087_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,55,879,14.218182
14,14,PD001_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,234,73,260,3.205479
17,17,14_133_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5582,2459,4092,2.270028
21,21,11_063_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3678,2415,1056,1.522981
24,24,PD041_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1205,572,672,2.106643
26,26,12_060_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5144,1708,2256,3.011710
30,30,PD013_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,507,318,736,1.594340


In [6]:
annotations_df[annotations_df["wm_gm_ratio"]<0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.474151
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332
12,12,PD110_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,292,1976,841,0.147773
19,19,14_073_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2962,6405,1320,0.462451
25,25,PD133_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,322,797,844,0.404015
27,27,14_075_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1920,4741,3084,0.404978
28,28,PD017_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1382,2897,20,0.477045


PD001
14_087
15_007
13_177

## Augmentation Functions

In [21]:
def augment_image(img_path_folder, filename, prefix):
    img = cv2.imread(os.path.join(img_path_folder,filename))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = A.Compose([
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
        A.RandomCrop(height=256, width=256,p=0.5),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5)])
    aug_img = transform(image=img)
    aug_img = Image.fromarray(aug_img["image"], "RGB")
    aug_img.save(os.path.join(img_path_folder, prefix+filename))



def perform_augmentation(path_to_folder, prefix):
    listfiles = os.listdir(path_to_folder)
    for filename in listfiles:
        augment_image(path_to_folder, filename, prefix)
    print("Augmentation done for image:",path_to_folder)


workers=4
def perform_parallel_augmentation(path_to_folder, prefix):
    listfiles = os.listdir(path_to_folder)
    #listfiles.remove('.DS_Store')
    exe = ThreadPoolExecutor(max_workers=workers)
    futures = [exe.submit(augment_image, path_to_folder, filename, prefix) for i, (filename) in enumerate(listfiles)]
    done, not_done = wait(futures, return_when=ALL_COMPLETED)
    exe.shutdown()



## Count All Crops including Augmented Images

In [7]:
def count_annotations_flag(filename, label):
    path = "/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/WM_images"
    annotated_images = os.listdir(path)
    annotated_images.remove('.DS_Store')
    filename = filename.replace(".geojson","")
    if filename in annotated_images:
        if os.path.exists(os.path.join(path,filename, label)):
            items = os.listdir(os.path.join(path,filename, label))
            #items = [len(os.listdir(os.path.join(path,filename,i))) for i in wgm_dir]
            return len(items)
        else :
            print("folder does not exist")
            return 0
    else:
        return None

# Iteration 1

In [53]:
indices = annotations_df[annotations_df["Aug1_wm_gm_ratio"]==annotations_df["wm_gm_ratio"]].index
for index in indices:
    path=None
    if annotations_df.iloc[index]["wm_gm_ratio"]<=0.5:
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["wm_gm_ratio"])
        # Add transformed White Matter images
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/White/"
        #perform_augmentation(path_to_wm, "aug1_")

    if annotations_df.iloc[index]["wm_gm_ratio"]>=1.5:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["wm_gm_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
        
    if path!=None:
        perform_augmentation(path,"aug1_")

14_073_CG_aSyn_x200.svs.geojson 0.4624512099921936
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_073_CG_aSyn_x200.svs/White/
11_063_CG_aSyn_x200.svs.geojson 1.5229813664596272
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/11_063_CG_aSyn_x200.svs/grey/
PD041_Syn1_CG.svs.geojson 2.1066433566433567
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD041_Syn1_CG.svs/grey/
PD133_Syn1_CG.svs.geojson 0.40401505646173147
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD133_Syn1_CG.svs/White/
12_060_CG_aSyn_x200.svs.geojson 3.011709601873536
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/12_060_CG_aSyn_x200.svs/grey/
14_075_CG_aSyn_x200.svs.geojson 0.40497785277367643
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/nps

In [ ]:
path_to_gm = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
perform_parallel_augmentation(path_to_gm,"aug1_")

In [8]:
annotations_df["Aug1_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug1_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
#annotations_df["Aug1_bg_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg"))

In [9]:
annotations_df["Aug1_wm_gm_ratio"]=annotations_df["Aug1_WM_count"]/annotations_df["Aug1_GM_count"]

In [12]:
annotations_df[annotations_df["Aug1_wm_gm_ratio"]==annotations_df["wm_gm_ratio"]].index

Int64Index([0, 4, 7, 8, 10, 11, 15, 16, 18, 20, 22, 23, 29], dtype='int64')

In [11]:
annotations_df

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio
0,0,14_148_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2253,3399,4110,0.662842,2253,3399,0.662842
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.474151,642,677,0.948301
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770
4,4,PD090_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,960,1879,1018,0.510910,960,1879,0.510910
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332,4878,5416,0.900665
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,2.446171,2204,1802,1.223085
7,7,14_053_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2189,3402,2293,0.643445,2189,3402,0.643445
8,8,PD079_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,836,1326,1020,0.630468,836,1326,0.630468
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912,3059,1574,1.943456


In [13]:
annotations_df[annotations_df["Aug1_wm_gm_ratio"]!=annotations_df["wm_gm_ratio"]]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.474151,642,677,0.948301
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332,4878,5416,0.900665
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,2.446171,2204,1802,1.223085
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912,3059,1574,1.943456
12,12,PD110_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,292,1976,841,0.147773,584,1976,0.295547
13,13,14_087_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,55,879,14.218182,782,110,7.109091
14,14,PD001_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,234,73,260,3.205479,234,146,1.602740
17,17,14_133_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5582,2459,4092,2.270028,5582,2748,2.031295


In [37]:
annotations_df[annotations_df["Aug1_wm_gm_ratio"]==annotations_df["wm_gm_ratio"]].index

Int64Index([ 0,  4,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
            22, 23, 24, 25, 26, 27, 28, 29, 30],
           dtype='int64')

In [14]:
annotations_df[annotations_df["Aug1_wm_gm_ratio"]<=0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio
12,12,PD110_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,292,1976,841,0.147773,584,1976,0.295547


In [15]:
annotations_df[annotations_df["Aug1_wm_gm_ratio"]>=1.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912,3059,1574,1.943456
13,13,14_087_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,55,879,14.218182,782,110,7.109091
14,14,PD001_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,234,73,260,3.205479,234,146,1.602740
17,17,14_133_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5582,2459,4092,2.270028,5582,2748,2.031295
26,26,12_060_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5144,1708,2256,3.011710,5144,3416,1.505855


# Iteration 2

In [16]:
for index in annotations_df.index:
    path=None
    if annotations_df.iloc[index]["Aug1_wm_gm_ratio"]<=0.5:
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_gm_ratio"])
        # Add transformed White Matter images
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/White/"
        #perform_augmentation(path_to_wm, "aug2_")

    if annotations_df.iloc[index]["Aug1_wm_gm_ratio"]>=1.5:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_gm_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
        
    if path!=None:
        perform_augmentation(path,"aug2_")

15_007_CG_aSyn_x200.svs.geojson 6.028846153846154
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/15_007_CG_aSyn_x200.svs/grey/
13_177_CG_aSyn_x200.svs.geojson 1.9434561626429478
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/13_177_CG_aSyn_x200.svs/grey/
PD110_Syn1_CG.svs.geojson 0.29554655870445345
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD110_Syn1_CG.svs/White/
14_087_CG_aSyn_x200.svs.geojson 7.109090909090909
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_087_CG_aSyn_x200.svs/grey/
PD001_Syn1_CG.svs.geojson 1.6027397260273972
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD001_Syn1_CG.svs/grey/
14_133_CG_aSyn_x200.svs.geojson 2.0312954876273652
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_

In [17]:
annotations_df["Aug2_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug2_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["Aug2_wm_gm_ratio"]=annotations_df["Aug2_WM_count"]/annotations_df["Aug2_GM_count"]

In [19]:
annotations_df[annotations_df["Aug2_wm_gm_ratio"]<=0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio


In [20]:
annotations_df[annotations_df["Aug2_wm_gm_ratio"]>=1.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846,1254,416,3.014423
13,13,14_087_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,55,879,14.218182,782,110,7.109091,782,220,3.554545


## Iteration 3

In [21]:
for index in annotations_df.index:
    path=None
    if annotations_df.iloc[index]["Aug2_wm_gm_ratio"]<=0.5:
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_gm_ratio"])
        # Add transformed White Matter images
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/White/"
        #perform_augmentation(path_to_wm, "aug2_")

    if annotations_df.iloc[index]["Aug2_wm_gm_ratio"]>=1.5:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_gm_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
        
    if path!=None:
        perform_augmentation(path,"aug3_")

15_007_CG_aSyn_x200.svs.geojson 6.028846153846154
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/15_007_CG_aSyn_x200.svs/grey/
14_087_CG_aSyn_x200.svs.geojson 7.109090909090909
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_087_CG_aSyn_x200.svs/grey/


In [22]:
annotations_df["Aug3_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug3_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["Aug3_wm_gm_ratio"]=annotations_df["Aug3_WM_count"]/annotations_df["Aug3_GM_count"]

In [24]:
annotations_df[annotations_df["Aug3_wm_gm_ratio"]<=0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio


In [23]:
annotations_df[annotations_df["Aug3_wm_gm_ratio"]>=1.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846,1254,416,3.014423,1254,832,1.507212
13,13,14_087_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,782,55,879,14.218182,782,110,7.109091,782,220,3.554545,782,440,1.777273


In [25]:
annotations_df

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio
0,0,14_148_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2253,3399,4110,0.662842,2253,3399,0.662842,2253,3399,0.662842,2253,3399,0.662842
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.474151,642,677,0.948301,642,677,0.948301,642,677,0.948301
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846,1254,416,3.014423,1254,832,1.507212
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770,408,678,0.601770,408,678,0.601770
4,4,PD090_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,960,1879,1018,0.510910,960,1879,0.510910,960,1879,0.510910,960,1879,0.510910
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332,4878,5416,0.900665,4878,5416,0.900665,4878,5416,0.900665
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,2.446171,2204,1802,1.223085,2204,1802,1.223085,2204,1802,1.223085
7,7,14_053_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2189,3402,2293,0.643445,2189,3402,0.643445,2189,3402,0.643445,2189,3402,0.643445
8,8,PD079_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,836,1326,1020,0.630468,836,1326,0.630468,836,1326,0.630468,836,1326,0.630468
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912,3059,1574,1.943456,3059,3148,0.971728,3059,3148,0.971728


In [26]:
annotations_df.to_csv("Augmented_Imgs_WM_GM_count.csv")

In [31]:
annotations_df["wm_bg_ratio"] = annotations_df["Aug3_WM_count"]/annotations_df["bg_count"]
annotations_df["gm_bg_ratio"] = annotations_df["Aug3_GM_count"]/annotations_df["bg_count"]

In [32]:
annotations_df[annotations_df["wm_bg_ratio"]>=1.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio,wm_bg_ratio,gm_bg_ratio
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332,4878,5416,0.900665,4878,5416,0.900665,4878,5416,0.900665,4.839286,5.373016
15,15,PD034_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2417,2348,976,1.029387,2417,2348,1.029387,2417,2348,1.029387,2417,2348,1.029387,2.476434,2.405738
18,18,14_036_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3903,3965,1364,0.984363,3903,3965,0.984363,3903,3965,0.984363,3903,3965,0.984363,2.861437,2.906891
19,19,14_073_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2962,6405,1320,0.462451,5924,6405,0.924902,5924,6405,0.924902,5924,6405,0.924902,4.487879,4.852273
21,21,11_063_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3678,2415,1056,1.522981,3678,4830,0.761491,3678,4830,0.761491,3678,4830,0.761491,3.482955,4.573864
23,23,14_153_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,4159,2776,2574,1.498199,4159,2776,1.498199,4159,2776,1.498199,4159,2776,1.498199,1.615773,1.078477
24,24,PD041_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1205,572,672,2.106643,1205,1144,1.053322,1205,1144,1.053322,1205,1144,1.053322,1.793155,1.702381
26,26,12_060_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,5144,1708,2256,3.011710,5144,3416,1.505855,5144,6832,0.752927,5144,6832,0.752927,2.280142,3.028369
28,28,PD017_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1382,2897,20,0.477045,2764,2897,0.954090,2764,2897,0.954090,2764,2897,0.954090,138.200000,144.850000


In [33]:
annotations_df[annotations_df["wm_bg_ratio"]<=0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio,wm_bg_ratio,gm_bg_ratio
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770,408,678,0.601770,408,678,0.601770,0.299120,0.497067
11,11,15_005_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1959,3493,4095,0.560836,1959,3493,0.560836,1959,3493,0.560836,1959,3493,0.560836,0.478388,0.852991
20,20,PD002_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2553,2505,5512,1.019162,2553,2505,1.019162,2553,2505,1.019162,2553,2505,1.019162,0.463171,0.454463
29,29,PD061_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1024,775,3406,1.321290,1024,775,1.321290,1024,775,1.321290,1024,775,1.321290,0.300646,0.227540


In [34]:
annotations_df[annotations_df["gm_bg_ratio"]<=0.5]

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,Aug2_WM_count,Aug2_GM_count,Aug2_wm_gm_ratio,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio,wm_bg_ratio,gm_bg_ratio
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846,1254,416,3.014423,1254,832,1.507212,0.645062,0.427984
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770,408,678,0.601770,408,678,0.601770,0.299120,0.497067
20,20,PD002_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2553,2505,5512,1.019162,2553,2505,1.019162,2553,2505,1.019162,2553,2505,1.019162,0.463171,0.454463
29,29,PD061_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1024,775,3406,1.321290,1024,775,1.321290,1024,775,1.321290,1024,775,1.321290,0.300646,0.227540


## Iteration 4

In [40]:
for index in annotations_df.index:
    path=None
    if annotations_df.iloc[index]["wm_bg_ratio"]<=0.5:
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["wm_bg_ratio"])
        # Add transformed White Matter images
        if annotations_df.iloc[index]["wm_bg_ratio"]<annotations_df.iloc[index]["gm_bg_ratio"]:
            path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/White/"
        else:
            path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
        #perform_augmentation(path_to_wm, "aug2_")

    if annotations_df.iloc[index]["Aug1_wm_gm_ratio"]>=1.5:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_gm_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/bg/"
        
    if path!=None:
        if path.split("/")[-2]=="bg":
            perform_augmentation(path,"aug1_")
        else:
            perform_augmentation(path,"aug4_")

15_007_CG_aSyn_x200.svs.geojson 6.028846153846154
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/15_007_CG_aSyn_x200.svs/bg/
PD131_Syn1_CG.svs.geojson 0.2991202346041056
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD131_Syn1_CG.svs/White/
13_177_CG_aSyn_x200.svs.geojson 1.9434561626429478
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/13_177_CG_aSyn_x200.svs/bg/
15_005_CG_aSyn_x200.svs.geojson 0.4783882783882784
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/15_005_CG_aSyn_x200.svs/White/
14_087_CG_aSyn_x200.svs.geojson 7.109090909090909
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_087_CG_aSyn_x200.svs/bg/
PD001_Syn1_CG.svs.geojson 1.6027397260273972
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_

In [41]:
annotations_df["Aug4_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug4_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["Aug1_BG_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg"))
#annotations_df["Aug3_wm_gm_ratio"]=annotations_df["Aug3_WM_count"]/annotations_df["Aug3_GM_count"]

In [42]:
annotations_df["Aug1_wm_bg_ratio"] = annotations_df["Aug4_WM_count"]/annotations_df["Aug1_BG_count"]
annotations_df["Aug1_gm_bg_ratio"] = annotations_df["Aug4_GM_count"]/annotations_df["Aug1_BG_count"]

In [43]:
annotations_df

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug1_WM_count,Aug1_GM_count,Aug1_wm_gm_ratio,...,Aug3_WM_count,Aug3_GM_count,Aug3_wm_gm_ratio,wm_bg_ratio,gm_bg_ratio,Aug4_WM_count,Aug4_GM_count,Aug1_BG_count,Aug1_wm_bg_ratio,Aug1_gm_bg_ratio
0,0,14_148_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2253,3399,4110,0.662842,2253,3399,0.662842,...,2253,3399,0.662842,0.548175,0.827007,2253,3399,4110,0.548175,0.827007
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.474151,642,677,0.948301,...,642,677,0.948301,0.506309,0.533912,642,677,1268,0.506309,0.533912
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,12.057692,1254,208,6.028846,...,1254,832,1.507212,0.645062,0.427984,1254,832,3888,0.322531,0.213992
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,0.300885,408,678,0.601770,...,408,678,0.601770,0.299120,0.497067,816,678,1364,0.598240,0.497067
4,4,PD090_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,960,1879,1018,0.510910,960,1879,0.510910,...,960,1879,0.510910,0.943026,1.845776,960,1879,1018,0.943026,1.845776
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.450332,4878,5416,0.900665,...,4878,5416,0.900665,4.839286,5.373016,4878,5416,1008,4.839286,5.373016
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,2.446171,2204,1802,1.223085,...,2204,1802,1.223085,0.762366,0.623314,2204,1802,2891,0.762366,0.623314
7,7,14_053_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2189,3402,2293,0.643445,2189,3402,0.643445,...,2189,3402,0.643445,0.954645,1.483646,2189,3402,2293,0.954645,1.483646
8,8,PD079_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,836,1326,1020,0.630468,836,1326,0.630468,...,836,1326,0.630468,0.819608,1.300000,836,1326,1020,0.819608,1.300000
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,3.886912,3059,1574,1.943456,...,3059,3148,0.971728,1.263006,1.299752,3059,3148,4844,0.631503,0.649876


In [44]:
annotations_df.sum()

Unnamed: 0                                                        465
filename            14_148_CG_aSyn_x200.svs.geojsonPD130_Syn1_CG.s...
filepath            /gladstone/finkbeiner/steve/work/data/npsad_da...
WM_count                                                        60801
GM_count                                                        64244
bg_count                                                        58487
wm_gm_ratio                                                 61.297929
Aug1_WM_count                                                   70643
Aug1_GM_count                                                   71466
Aug1_wm_gm_ratio                                            42.155771
Aug2_WM_count                                                   71227
Aug2_GM_count                                                   79668
Aug2_wm_gm_ratio                                            32.340676
Aug3_WM_count                                                   71227
Aug3_GM_count       

## Iteration 5

In [45]:
for index in annotations_df.index:
    path=None
    if annotations_df.iloc[index]["Aug1_wm_bg_ratio"]<=0.5:
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["wm_bg_ratio"])
        # Add transformed White Matter images
        if annotations_df.iloc[index]["Aug1_wm_bg_ratio"]<annotations_df.iloc[index]["Aug1_gm_bg_ratio"]:
            path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/White/"
        else:
            path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/grey/"
        #perform_augmentation(path_to_wm, "aug2_")

    if annotations_df.iloc[index]["Aug1_wm_bg_ratio"]>=1.5:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["Aug1_wm_bg_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/bg/"
        
    if path!=None:
        if path.split("/")[-2]=="bg":
            perform_augmentation(path,"aug2_")
        else:
            perform_augmentation(path,"aug5_")

15_007_CG_aSyn_x200.svs.geojson 0.6450617283950617
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/15_007_CG_aSyn_x200.svs/grey/
13_131_CG_aSyn_x200.svs.geojson 4.839285714285714
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/13_131_CG_aSyn_x200.svs/bg/
14_087_CG_aSyn_x200.svs.geojson 0.8896473265073948
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_087_CG_aSyn_x200.svs/grey/
PD001_Syn1_CG.svs.geojson 0.9
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD001_Syn1_CG.svs/White/
PD034_Syn1_CG.svs.geojson 2.4764344262295084
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD034_Syn1_CG.svs/bg/
14_036_CG_aSyn_x200.svs.geojson 2.8614369501466275
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_

In [8]:
annotations_df["Aug_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["Aug_BG_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg"))

In [10]:
annotations_df.sum()

Unnamed: 0                                                    465
filename        14_148_CG_aSyn_x200.svs.geojsonPD130_Syn1_CG.s...
filepath        /gladstone/finkbeiner/steve/work/data/npsad_da...
WM_count                                                    60801
GM_count                                                    64244
bg_count                                                    58487
wm_gm_ratio                                             61.297929
Aug_WM_count                                                75314
Aug_GM_count                                                84856
Aug_BG_count                                                75008
dtype: object

In [16]:
annotations_df["wm_bg_ratio"]  = annotations_df["Aug_WM_count"]/annotations_df["Aug_BG_count"]

In [17]:
annotations_df["gm_bg_ratio"]=annotations_df["Aug_GM_count"]/annotations_df["Aug_BG_count"]
annotations_df["wm_gm_ratio"]=annotations_df["Aug_WM_count"]/annotations_df["Aug_GM_count"]

In [18]:
annotations_df

,Unnamed: 0,filename,filepath,WM_count,GM_count,bg_count,wm_gm_ratio,Aug_WM_count,Aug_GM_count,Aug_BG_count,wm_bg_ratio,gm_bg_ratio
0,0,14_148_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2253,3399,4110,0.662842,2253,3399,4110,0.548175,0.827007
1,1,PD130_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,321,677,1268,0.948301,642,677,1268,0.506309,0.533912
2,2,15_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,1254,104,1944,0.753606,1254,1664,3888,0.322531,0.427984
3,3,PD131_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,204,678,1364,1.203540,816,678,1364,0.598240,0.497067
4,4,PD090_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,960,1879,1018,0.510910,960,1879,1018,0.943026,1.845776
5,5,13_131_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2439,5416,1008,0.900665,4878,5416,2016,2.419643,2.686508
6,6,12_007_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2204,901,2891,1.223085,2204,1802,2891,0.762366,0.623314
7,7,14_053_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,2189,3402,2293,0.643445,2189,3402,2293,0.954645,1.483646
8,8,PD079_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,836,1326,1020,0.630468,836,1326,1020,0.819608,1.300000
9,9,13_177_CG_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,3059,787,2422,0.971728,3059,3148,4844,0.631503,0.649876


In [22]:
for index in annotations_df.index:
    path=None
    if annotations_df.iloc[index]["wm_bg_ratio"]>=2:
        # Add transformed Grey Matter images
        print(annotations_df.iloc[index]["filename"], annotations_df.iloc[index]["wm_bg_ratio"])
        path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/"+annotations_df.iloc[index]["filename"].replace(".geojson","")+"/bg/"
        
    if path!=None:
        if path.split("/")[-2]=="bg":
            perform_augmentation(path,"aug2_")

13_131_CG_aSyn_x200.svs.geojson 2.419642857142857
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/13_131_CG_aSyn_x200.svs/bg/
14_073_CG_aSyn_x200.svs.geojson 2.243939393939394
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/14_073_CG_aSyn_x200.svs/bg/
11_063_CG_aSyn_x200.svs.geojson 3.4829545454545454
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/11_063_CG_aSyn_x200.svs/bg/
PD017_Syn1_CG.svs.geojson 138.2
Augmentation done for image: /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_images/PD017_Syn1_CG.svs/bg/


In [24]:
annotations_df["Aug_WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["Aug_GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["Aug_BG_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg"))

In [25]:
annotations_df.sum()

Unnamed: 0                                                    465
filename        14_148_CG_aSyn_x200.svs.geojsonPD130_Syn1_CG.s...
filepath        /gladstone/finkbeiner/steve/work/data/npsad_da...
WM_count                                                    60801
GM_count                                                    64244
bg_count                                                    58487
wm_gm_ratio                                             28.504306
Aug_WM_count                                                75649
Aug_GM_count                                                84856
Aug_BG_count                                                78412
wm_bg_ratio                                            170.544753
gm_bg_ratio                                            182.327434
dtype: object

In [27]:
annotations_df.to_csv("WM_Annotations_with_augmented_counts.csv")